## Installs

In [ ]:
!pip install -U torch torchvision
!pip install transformers datasets tqdm pandas scipy

In [ ]:
!pip install --force-reinstall --no-cache-dir scipy # Only needed within runpod environment
!pip install --force-reinstall --no-cache-dir typing_extensions==4.11.0
!pip uninstall -y Pillow
!pip install Pillow
!pip install numpy==1.26.4

In [ ]:
## Sometimes needed in runpod to make sure it goes to the network volumne
import os

os.environ["HF_DATASETS_CACHE"] = "/workspace/hf_cache"
os.environ["TRANSFORMERS_CACHE"] = "/workspace/hf_cache"
os.environ["HF_HOME"] = "/workspace/hf_home"

In [ ]:
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm import tqdm
from transformers import AutoModel
from datasets import load_from_disk, concatenate_datasets
import pandas as pd
import numpy as np
import os
from collections import defaultdict
import copy
from typing import Optional

## Configuration

In [ ]:
dataset_names = []
num_class = []
size_nums = []
model_name = "DINOv3"
transformation = "Standard"
indices = [i for i in range(12)]
device = "cuda" if torch.cuda.is_available() else "cpu"

# https://huggingface.co/collections/facebook/dinov3-68924841bd6b561778e31009
pretrained_loading_name = "facebook/dinov3-vitb16-pretrain-lvd1689m" # https://huggingface.co/facebook/dinov3-vitb16-pretrain-lvd1689m
refer = AutoModel.from_pretrained(
    pretrained_loading_name,
    device_map="auto",
)
embedding_space_size = refer.config.hidden_size

## Class Prep

In [ ]:
class Hooks(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.CLS = []
    
    def forward(self, pixel_values):
        self.CLS = []
        pixel_values = pixel_values.to(self.model.embeddings.path_embeddings.weight.dtype)
        hidden_states = self.model.embeddings(pixel_values)
        position_embeddings = self.model.rope_embedings(pixel_values)

        for layer_module in self.model.layer:
            hidden_states = layer_module(hidden_states, position_embeddings=position_embeddings)
            self.CLS.append(hidden_states[:, 0, :])
        
        return self.CLS

In [ ]:
# https://github.com/huggingface/transformers/blob/main/src/transformers/models/dinov3_vit/modeling_dinov3_vit.py
class Augmented(torch.nn.Module):
    def __init__(self, model, embedding_space_dimension, num_classes, classifier_head=None, transformation_layer=-1, W=None):
        super().__init__()
        self.model = model
        self.embedding_space_dimension = self.embedding_space_dimension
        self.num_classes = num_classes
        self.classifier_head = self.classifier_head if classifier_head is not None else torch.nn.Linear(self.embedding_space_dimension, self.num_classes)
        self.transformation_layer = transformation_layer
        self.W = torch.from_numpy(W.astype(np.float32)).to(device) if W is not None else None
    
    def forward(self, pixel_values):
        pixel_values = pixel_values.to(self.model.embeddings.path_embeddings.weight.dtype)
        hidden_states = self.model.embeddings(pixel_values)
        position_embeddings = self.model.rope_embedings(pixel_values)

        for i, layer_module in enumerate(self.model.layer):
            hidden_states = layer_module(hidden_states, position_embeddings=position_embeddings)
            if i == self.transformation_layer:
                if self.W is None:
                    self.W = torch.eye(self.embedding_space_dimension)
                hidden_states = hidden_states @ self.W
        
        pre_norm_pooled_output = hidden_states[:, 0, :]
        sequence_output = self.norm(hidden_states)
        post_norm_pooled_output = sequence_output[:, 0, :]

        logits = self.classifier_head(post_norm_pooled_output)

        return {
            "logits": logits, 
            "pre_norm_pooled_output": pre_norm_pooled_output,
            "post_norm_pooled_output": post_norm_pooled_output,
        }

## Evaluation Functions

In [ ]:
def extract_vectors(indices, train_dataset, base_model, fine_tuned_model):
    Z0 = {i: [] for i in indices}
    Z1 = []

    with torch.no_grad():
        for batch in tqdm(train_dataset, desc="Extracting Vectors"):
            pixel_values = batch["images"].to(device, non_blocking=True)
            base_output = base_model(pixel_values)
            fine_tuned_output = fine_tuned_model(pixel_values)

            for i in indices:
                Z0[i].append(base_output[i].float().cpu())
            fine_tuned_output.append(fine_tuned_output[-1].float().cpu())
    
    return Z0, Z1

In [ ]:
def augment_models(indices, transformation, num_classes, embedding_space_dimension, reference, fine_tuned, W=None, linear_probe=False):
    if W is None:
        W = [None for i in indices]
    augmented_models = []
    for i in indices:
        if transformation in ["Standard", "Base_Fine_Tuned_Head"]:
            model = Augmented(model=copy.deepcopy(reference), embedding_space_dimension=embedding_space_dimension, num_classes=num_classes, classifier_head=copy.deepcopy(fine_tuned.classifier_head), transformation=i, W=W[i])
        elif transformation == "Linear_Probe":
            model = Augmented(model=copy.deepcopy(refer), embedding_space_dimension=embedding_space_dimension, num_classes=num_classes, classifier_head=copy.deepcopy(linear_probe.classifier_head), transformation=i, W=W[i])
        model = model.eval().to(device)
        augmented_models[i] = model
    return augmented_models

In [ ]:
def cosine_similarity():

In [ ]:
def evaluate(indices, test_dataset, augmented_models):
    augmented_accuracy = {i: 0 for i in indices}
    augmented_pre_norm_cls_cosine_similarity = {i: [] for i in indices}
    augmented_post_norm_cls_cosine_similarity = {i: [] for i in indices}
    total = 0

    with torch.no_grad():
        for batch in tqdm(test_dataset, desc="Test-Set Evaluation"):
            pixel_values = batch["images"].to(device, non_blocking=True)
            labels = batch["label"].to(device, non_blocking=True)
            total += labels.size(0)

            for i in indices:
                prediction = augmented_models[i](pixel_values)
                logits = prediction["logits"].argmax(dim=1)
                pre_norm_cls_token = prediction["pre_norm_pooled_output"]
                post_norm_cls_token = prediction["post_norm_pooled_output"]

                augmented_accuracy[i] += (logits == labels).sum().item()
    
    for i in indices:
        augmented_accuracy[i] = augmented_accuracy[i] / total
        print(f"Augmented Layer {i} - Accuracy {augmented_accuracy[i]} | Pre-Norm CLS Cosine Similarity - {augmented_pre_norm_cls_cosine_similarity} | Post-Norm CLS Cosine Similarity - {augmented_post_norm_cls_cosine_similarity}")
    
    return {
        "Accuracy": augmented_accuracy, 
        "Pre_Norm_CLS_Cosine_Similarity": augmented_pre_norm_cls_cosine_similarity,
        "Post_Norm_CLS_Cosine_Similarity": augmented_post_norm_cls_cosine_similarity,
    }

In [ ]:
def save_result(indices, result_path, accuracy, pre_norm_cls_cosine_similarity, post_norm_cls_cosine_similarity, residuals=None, W=None):
    data = {
        "Classification_Accuracy": [accuracy[i] for i in indices],
        "Pre_Norm_CLS_Cosine_Similarity": [pre_norm_cls_cosine_similarity[i] for i in indices],
        "Post_Norm_CLS_Cosine_Similarity": [post_norm_cls_cosine_similarity[i] for i in indices],
    }

    if W is not None:
        data["W"] = [W[i] for i in indices]
    if residuals is not None:
        data["Residuals"] = [W[i] for i in indices]

    data_frame = pd.DataFrame(data, index=indices)
    data_frame.to_json(result_path, orient="records", indent=2)

## Evaluation Loop

In [ ]:
for name in range(0, len(dataset_names)):
    train = load_from_disk()
    val = load_from_disk()
    test = load_from_disk()

    # Sort Through Dataset by Class
    labels = train["label"]

    label_to_indices = defaultdict(list)

    for idx, label in enumerate(labels):
        label = int(label)
        label_to_indices[label].append(idx)

    filtered_train = {
        label: train.select(indices) for label, indices in label_to_indices.items()
    }

    for trial in range(1,6):
        result_path = f"./Results/{model_name}/Long_Training/{dataset_names}/{i}"
        os.makedirs(result_path, exist_ok=True)

        # Loading Models
        base = Augmented(copy.deepcopy(refer), embedding_space_dimension=embedding_space_size, num_classes=num_class[name])
        fine_tuned = Augmented(copy.deepcopy(refer), embedding_space_dimension=embedding_space_size, num_classes=num_class[name])
        fine_tuned.load_state_dict(torch.load(f"{model_name}/Models/Fine_Tuned/{dataset_names[name]}/"))
        linear_probe = Augmented(copy.deepcopy(refer), embedding_space_dimension=embedding_space_size, num_classes=num_class[name])
        linear_probe.load_state_dict(torch.load(f"{model_name}/Models/Linear_Probe/{dataset_names[name]}/"))

        base_hooks = Hooks(copy.deepcopy(refer))
        fine_tuned_hooks = Hooks(copy.deepcopy(fine_tuned.model))

        # All Class-Specific Train-Set Embeddings
        # Shape (num_class, indices)
        Z0 = {}
        Z1 = {}
        for i in range(num_class[name]):
            class_subset = concatenate_datasets(filtered_train[i])
            train_loader = DataLoader(class_subset, batch_size=64, shuffle=True, collate_fn=collate_fn, pin_memory=True, num_workers=8, persistent_workers=True)
            Z0[i], Z1[i] = extract_vectors(indices, train_loader, base_hooks, fine_tuned_hooks)
            Z1[i] - torch.cat(Z1).cpu().numpy()
            for j in indices:
                Z0[i][j] = torch.cat(Z0[i][j].cpu().numpy())

        # Evaluations
        for number in range(0, len(size_nums)):
            num_img_per_class = size_nums[number]

            Z0_subset = {[] for i in indices} # (layer, images)
            W = {}
            resid = {}

            sorted = []
            for i in range(num_class[name]):
                indice_end = min(num_img_per_class, len(filtered_train[i]))
                for j in indices:
                    Z0[i].append(Z0[i][j][0:indice_end])
            train_size = len(sorted)

            for i in indices:
                W[i], resid[i], _, _ = np.linalg.lstsq(sorted, Z1, rcond=None)
            
            print(f"Results for {model_name} on {dataset_names[name]}: {train_size} Training Images ({num_img_per_class} images/label)")
            augmented_models = augment_models(indices, transformation, num_class[name], embedding_space_size, refer, fine_tuned, W=W, linear_probe=False)
            results = evaluate(indices, test_dataset, augmented_models)
            save_result(indices, result_path, results["Accuracy"], results["Pre_Norm_CLS_Cosine_Similarity"], results["Post_Norm_CLS_Cosine_Similarity"], residuals=resid, W=None)
        
        print("Ablation Tests")